[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saketkc/pysradb/blob/develop/notebooks/09.Metadata_Enrichment_with_LLMs.ipynb)

# Metadata Enrichment with LLMs

pysradb now helps you 'enrich' your metadata frame by leveraging recent advancements in (fast) (S)LLMs. The parsed metadata can be fed to an LLM that is 'instructed' to enrich the metadata by returning 9 ontology-based fields::
`organ`, `tissue`, `anatomical_system`, `cell_type`, `disease`, `sex`, `development_stage`, `assay`, `organism`

There are two approaches in which this is possible:

- **LLMs** ([Requires Ollama](https://ollama.com/download)  - local, no API keys)
- **Embeddings** ([Using sentence-transformers](https://huggingface.co/sentence-transformers) with [BioLORD](https://arxiv.org/abs/2311.16075) embedding model)


In [1]:
# Install pysradb if not already installed
try:
    import pysradb

    print(f"pysradb {pysradb.__version__} is already installed")
except ImportError:
    print("Installing pysradb from GitHub...")
    import sys

    !{sys.executable} -m pip install -q git+https://github.com/saketkc/pysradb
    print("pysradb installed successfully!")

Installing pysradb from GitHub...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
pysradb installed successfully!


In [2]:

from pysradb import SRAweb
from pysradb.metadata_enrichment import create_metadata_extractor
from pysradb.search import GeoSearch



/usr/local/lib/python3.12/dist-packages/pysradb/utils.py:16: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Quick Start

One line enrichment!

**Prerequisites**: Install Ollama (https://ollama.ai) and pull a model: `ollama pull phi3`

The easiest way to enrich metadata is using the `enrich=True` parameter:

In [3]:
! sudo apt update -qq && sudo apt install -qq pciutils lshw systemd

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,413 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,288 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,479 kB]
Get:

In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
!nohup ollama serve > ollama.log 2>&1 &


NAME           ID              SIZE      MODIFIED               
phi3:latest    4f2222927938    2.2 GB    Less than a second ago    


In [8]:
!ollama pull phi3:latest && ollama list


NAME           ID              SIZE      MODIFIED               
phi3:latest    4f2222927938    2.2 GB    Less than a second ago    


In [9]:
from pysradb.sraweb import SRAweb

db = SRAweb()

df = db.metadata("GSE155673", detailed=True, enrich=True)

cols = [
    "sample_title",
    "sex",
    "guessed_sex",
    "tissue",
    "guessed_tissue",
    "guessed_organ",
    "guessed_anatomical_system",
    "guessed_cell_type",
    "guessed_organ",
    "guessed_tissue",
    "guessed_disease",
]
display(df[cols])

Enriching metadata:   0%|          | 0/24 [00:00<?, ?row/s]

,sample_title,sex,guessed_sex,tissue,guessed_tissue,guessed_organ,guessed_anatomical_system,guessed_cell_type,guessed_organ,guessed_tissue,guessed_disease
0,cov_01_RNA,F,female,<NA>,peripheral blood,blood,immune system,pbmc,blood,peripheral blood,covid-19
1,cov_01_antibody,F,female,<NA>,peripheral blood,blood,immune system,pbmc,blood,peripheral blood,covid-19
2,cov_02_RNA,F,female,<NA>,peripheral blood,blood,immune system,pbmc,blood,peripheral blood,covid-19
3,cov_02_antibody,F,female,<NA>,peripheral blood,blood,immune system,pbmc,blood,peripheral blood,covid-19
4,cov_03_RNA,F,female,<NA>,peripheral blood,blood,immune system,pbmc,blood,peripheral blood,covid-19
5,cov_03_antibody,F,female,<NA>,peripheral blood,blood,immune system,pbmc,blood,peripheral blood,covid-19
6,cov_04_RNA,M,male,<NA>,peripheral blood,blood,immune system,pbmc,blood,peripheral blood,covid-19
7,cov_04_antibody,M,male,<NA>,peripheral blood,blood,immune system,pbmc,blood,peripheral blood,covid-19
8,cov_07_RNA,F,female,<NA>,peripheral blood,blood,immune system,pbmc,blood,peripheral blood,healthy
9,cov_07_antibody,F,female,<NA>,peripheral blood,blood,immune system,pbmc,blood,peripheral blood,healthy


---

## Manual Enrichment


For more control, you can manually create extractors and enrich DataFrames:

### Manual LLM-Based Enrichment


In [10]:
# Create LLM extractor
extractor_llm = create_metadata_extractor(method="llm", backend="ollama/phi3")

# Test extraction
text = (
    "Single-cell RNA-seq of CD4+ T cells from breast cancer patients. sex: F. age: 55"
)
result = extractor_llm.extract_metadata(text)

print("Extracted metadata:")
for key, value in result.items():
    if value != "Unknown":
        print(f"  {key}: {value}")

Extracted metadata:
  organ: breast
  anatomical_system: digestive system
  cell_type: cd4+ t cell
  disease: cancer
  sex: female
  development_stage: adult
  assay: single-cell rna seq
  organism: homo sapiens


In [11]:
if not df.empty:
    df_enriched = extractor_llm.enrich_dataframe(
        df.head(3), text_column="sample_title", prefix="guessed_"
    )

    cols = ["sample_title"] + [
        c for c in df_enriched.columns if c.startswith("guessed_")
    ]
    display(df_enriched[cols])

Enriching metadata:   0%|          | 0/3 [00:00<?, ?row/s]

,sample_title,guessed_organ,guessed_tissue,guessed_anatomical_system,guessed_cell_type,guessed_disease,guessed_sex,guessed_development_stage,guessed_assay,guessed_organism
0,cov_01_RNA,brain,brain tissue,nervous system,neuron,healthy,unknown,adult,rna-seq,homo sapiens
1,cov_01_antibody,unknown,unknown,unknown,pbmc,healthy,unknown,adult,rna-seq,homo sapiens
2,cov_02_RNA,brain,brain tissue,nervous system,neuron,healthy,unknown,adult,rna-seq,homo sapiens


### Manual Embedding-Based Enrichment

Faster than LLMs, works offline. Load comprehensive ontology reference (31K+ terms):

In [12]:
# Load ontology reference (31K+ terms from UBERON, MONDO, CL)
from pysradb.metadata_enrichment import load_ontology_reference

ontology_ref = load_ontology_reference()
print(f"Loaded {sum(len(v) for v in ontology_ref.values()):,} ontology terms")

# Create embedding extractor
extractor_emb = create_metadata_extractor(
    method="embedding",
    model="FremyCompany/BioLORD-2023",
    reference_categories=ontology_ref,
)

text = "scRNA-seq of CD8+ T cells from melanoma patients"
result = extractor_emb.extract_metadata(text)
print(f"\nExtracted: {result}")

Loaded 698 ontology terms


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Extracted: {'organ': 'Unknown', 'tissue': 'Unknown', 'anatomical_system': 'Unknown', 'cell_type': 'CD8 T cell', 'disease': 'melanoma', 'sex': 'Unknown', 'development_stage': 'Unknown', 'assay': 'microRNA sequencing', 'organism': 'Unknown'}
